> Notebook-friendly copy of `part-I/1.5-pandas-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Import pandas as `pd` and numpy as `np`, seed any randomness with `default_rng`, and keep units in your column names.

## Exercise 1: A DataFrame and a column mean

Build a DataFrame with columns `temp_celsius = [18.2, 17.5, 19.1, 16.8]` and `discharge_m3s = [48.0, 51.2, 47.5, 53.1]`. Print the column dtypes and the mean temperature, rounded to two decimals.

In [ ]:
# Your solution here

## Exercise 2: Write and read a CSV with dates

Build a DataFrame with a `date` column (`"2024-06-01"`, `"2024-06-02"`, `"2024-06-03"`) and a `temp_celsius` column. Write it to `_files/obs.csv` without the index, then read it back with `parse_dates=["date"]` and print the dtypes.

In [ ]:
# Your solution here

## Exercise 3: Label, position, and mask

Given

```python
df = pd.DataFrame({"station": ["BAS", "LUG", "JFJ"], "temp_celsius": [18.0, 21.0, -1.0]},
                  index=["a", "b", "c"])
```

print the temperature at label `"b"`, the whole first row by position, and all rows whose temperature is below 0 °C.

In [ ]:
# Your solution here

## Exercise 4: `.isel` and `.sel` on a DataArray

The block below stores ten days of temperature at three stations twice: as an xarray `DataArray` with dimensions `time` and `station`, and as a DataFrame with the dates as rows and the stations as columns. Using `temp_da`, print:

1. every station's first day, by position with `.isel` — and the same row of `temp_df` with `.iloc`;
2. JFJ on 5 June, by label with `.sel` — and the same cell of `temp_df` with `.loc`;
3. LUG's last three days — `.isel` also accepts a list of positions;
4. the BAS reading nearest to 18:00 on 7 June, and the date it was taken. Select the station first and apply `method="nearest"` to the time alone: `.sel` applies the method to every dimension it is given, and the station names cannot be "nearest" to anything.

In a comment, say why item 4 returns the day it does, and what `temp_da.isel(station=0)` would return if the array had been built with its dimensions in the other order, `("station", "time")`.

```python
import xarray as xr

rng = np.random.default_rng(2)
dates = pd.date_range("2024-06-01", periods=10, freq="D")
temp_celsius = np.array([18.0, 21.0, -1.0]) + rng.normal(0, 1.5, size=(10, 3))   # (time, station)

temp_da = xr.DataArray(temp_celsius, dims=("time", "station"),
                       coords={"time": dates, "station": ["BAS", "LUG", "JFJ"]},
                       attrs={"units": "degC"})
temp_df = pd.DataFrame(temp_celsius, index=dates, columns=["BAS", "LUG", "JFJ"])
```

In [ ]:
# Your solution here

## Exercise 5: Resample and rolling

Build a 40-day daily Series indexed by date (`np.random.default_rng(0)`, mean 18, std 2). Print the monthly means and the first five values of the 3-day rolling mean, both rounded to two decimals.

In [ ]:
# Your solution here

## Exercise 6: Group and aggregate

Given

```python
df = pd.DataFrame({"station": ["BAS", "BAS", "LUG", "LUG"],
                   "temp_celsius": [18.0, 19.0, 21.0, 22.0]})
```

compute the mean temperature per station and print it as a dict.

In [ ]:
# Your solution here

## Exercise 7: Handle a gap three ways

Given `s = pd.Series([1.0, np.nan, np.nan, 4.0, 5.0])`, print the forward-filled series, the linearly interpolated series, and the NaN-skipping mean. Note in a comment why the three differ.

In [ ]:
# Your solution here

## Exercise 8: Join two tables

Given an observations table and a metadata table that share a `station` key, left-join the elevation onto the observations and print the result.

```python
obs = pd.DataFrame({"station": ["BAS", "LUG"], "temp_celsius": [18.0, 21.0]})
meta = pd.DataFrame({"station": ["BAS", "LUG"], "elevation_m": [316, 273]})
```

Then stack the row below — a third station that reports only temperature — onto the joined table with `concat`, and print the result. Which cell comes out missing, and why?

```python
jfj = pd.DataFrame({"station": ["JFJ"], "temp_celsius": [-1.0]})
```

In [ ]:
# Your solution here

## Exercise 9: A log-scale axis, and counting past a threshold

River discharge is a classic right-skewed variable: most days sit near base flow, with a long tail of rare, much larger flood events.

```python
rng = np.random.default_rng(1)
discharge_m3s = pd.Series(np.exp(rng.normal(3.4, 1.3, 300)))
```

1. Plot a histogram of `discharge_m3s` with `.plot(kind="hist")`, on default (linear) axes, then again with `logy=True`. Which view makes the long tail easier to read?
2. Build a boolean mask for discharge above 100 m³ s⁻¹ (a "flood" threshold), filter `discharge_m3s` with it, and print how many of the 300 values exceed it.

In [ ]:
# Your solution here

## Exercise 10: Filtered vs. unfiltered histogram

```python
rng = np.random.default_rng(0)
temp_celsius = pd.Series(rng.normal(18.0, 2.0, 200))
temp_celsius[:5] = -999.0     # a stuck sensor: five obviously invalid readings
```

1. Plot a histogram of the raw `temp_celsius` values with `.plot(kind="hist")`. What does the stuck sensor do to the plot?
2. Build a boolean mask that keeps only values above −50 °C, filter the Series with it, and plot the histogram again. In one comment, say which of the two plots you would trust.
3. Finally, mark the stuck readings in `temp_celsius` itself as missing: set every value below −50 °C to `np.nan` with a masked `.loc` assignment, then print the mean of the result next to the mean of the filtered Series from step 2. Why do the two agree?

In [ ]:
# Your solution here

## Exercise 11: Earthquake data analysis

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/usgs-2014-south-napa-earthquake.jpg" alt="A dirt track through a vineyard, broken along its length by a continuous ridge of pushed-up soil following the fault trace" width="800">

<em>Surface rupture from the magnitude 6.0 South Napa earthquake of 24 August 2014 — the kind of event a row in this catalog stands for. The continuous "mole track" running parallel to the strike of the fault indicates some east–west compression on top of the <a href="https://www.usgs.gov/faqs/what-fault-and-what-are-different-types">right-lateral faulting</a>; taken near Buhman Road. Photo <a href="https://www.usgs.gov/media/images/2014-south-napa-ca-m6-earthquake-august-24-6">USGS</a>, public domain.</em>


This exercise reviews the `pandas` fundamentals of this subchapter on one real catalog, covering how to

* open csv files
* manipulate dataframe indexes
* parse date columns
* examine basic dataframe statistics
* manipulate text columns and extract values
* plot dataframe contents using
  * bar charts
  * histograms
  * scatter plots

The data is a snapshot of the [USGS Earthquakes Database](https://earthquake.usgs.gov/earthquakes/search/): every event recorded worldwide in 2014, 120 108 rows.

You do not need to download the file yourself. The pre-supplied cell below fetches and caches it with the help of [pooch](https://www.fatiando.org/pooch/latest/) — there is no need to read pooch's documentation unless you want to — and stores the path to the file in the variable `datafile`.

In [ ]:
# Pre-supplied: download and cache the earthquake data.
import pooch

datafile = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2014.csv",
    known_hash="sha256:84d455fb96dc8f782fba4b5fbe56cb8970cab678f07c766fcba1b1c4674de1b1",
    fname="usgs_earthquakes_2014.csv",
    path=pooch.os_cache("mlees"),
)

**ℹ️ Beyond this subchapter**

The tools below were not demonstrated in 1.5's lecture. Each question links its documentation where
it is needed; they are collected here so you know when you are being asked to read a new page rather
than to recall one:

- [`nlargest`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html) — Q5
- [pandas' text data functions](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html), and [`Series.str.split`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html) in particular — Q6
- [`unique`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.unique.html) — Q7
- [`count`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html) and [`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) — Q9

The scatter plot and colorbar in Q11 are the ones from 1.4.6. Everything else the questions ask for
is from this subchapter's lecture: `read_csv` with date-parsed columns, `set_index`, `.head()`,
`.info()`, `.describe()`, boolean filtering, `.index` and `.values`, `kind="bar"`, `kind="hist"`, and
a logarithmic count axis.

**Q1) First, import `numpy`, `pandas` and `matplotlib`, and — optionally — set the display options.**

Hint: display options are documented [at this link](https://pandas.pydata.org/pandas-docs/stable/user_guide/options.html).

In [ ]:
# Import all libraries here

**Q2) Use pandas' `read_csv` function directly on `datafile` to open it as a `DataFrame`.**

Display the first few rows with `.head()`, and the column names, dtypes and non-null counts with
`.info()`. Check these tutorials if you have doubts about what the two functions do:
[`.head()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html)
and
[`.info()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html?highlight=info#pandas.DataFrame.info).

`.info()` will show you that the two date columns, `time` and `updated`, came in as plain strings
rather than as dates. They were not parsed automatically. What can you do about that?

In [ ]:
# Open the file as a pandas DataFrame, then display the first few rows and the DataFrame info

**Q3) Re-read the data in such a way that both date columns are identified as dates and the earthquake ID is used as the index.**

Check with `.head()` and `.info()` that it worked: `time` and `updated` should now be
`datetime64` columns, and `id` should be the index rather than a column.

Hint: the documentation for `read_csv` is [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html?highlight=read_csv).

In [ ]:
# Re-read the file, then check with head and info that it worked

**Q4) Use `describe` to get the basic statistics of all the columns.**

Hint: the documentation of `describe` is [at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html).

In [ ]:
# Use the describe function

**Q5) Use `nlargest` to get the top 20 earthquakes by magnitude.**

Hint: the documentation of `nlargest` is [at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html).

In [ ]:
# Use nlargest

**Q6) Extract the state or country using pandas' text data functions, and add it as a new column to the `DataFrame`.**

Examine the column titled `place`: it holds strings like `"26km S of Redoubt Volcano, Alaska"`, so
it carries both the local description and the state or country. Pull the second part out and store
it in a new column called `country`.

Hint 1: the documentation for pandas' text data functions is [here](https://pandas.pydata.org/pandas-docs/stable/text.html).

Hint 2: you will use `.split()` to extract the country names. The documentation of this function
can be found [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html?highlight=split#pandas.Series.str.split).

In [ ]:
# Extract the state or country, and add it as a new column to the DataFrame called country

**Q7) Display each unique value from the new `country` column.**

Hint: you may use the `unique` function documented [at this link](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.unique.html).

You should see an array of a couple of hundred country and state names. A plain split leaves the
space that followed the comma, so they come out as `array([' Alaska', ' Nevada', ...])` — worth
removing before you use them as labels.

In [ ]:
# Display unique values

**Q8) Create a filtered dataset that only has earthquakes larger than magnitude 4.**

Hint: print the table to see the name of the column containing the earthquake magnitude.

Check out this [link](https://www.geeksforgeeks.org/ways-to-filter-pandas-dataframe-by-column-values/)
to find examples of filtering by column values.

In [ ]:
# Filter the dataset based on the earthquakes' magnitudes

**Q9) Using the filtered dataset (magnitude > 4), count the number of earthquakes whose magnitudes are above 4 [Num 1], and count the number of earthquakes in each country or state [Num 2]. Make a bar chart of Num 2 for the top 5 locations with the most earthquakes.**

Hint 1: to get Num 1, pandas has a `count` function documented
[at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html).

Hint 2: check out the
[`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
function to get Num 2.

Then convert the first five rows of Num 2 into a `DataFrame` with two columns, `country` (text)
and `earthquake_num` (number), and recreate the bar chart below.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.5-target-top5-bar.png" alt="A bar chart of the number of magnitude-above-4 earthquakes in the five countries with the most of them" width="700">

<em>The five locations with the most magnitude-above-4 events in 2014.</em>

In [ ]:
# Count the >4 earthquakes, count them per country, then plot the top 5 as a bar chart

**Q10) Make a histogram for the distribution of the earthquakes' magnitudes.**

Hint: pandas has a histogram function documented
[at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html) and
matplotlib has one documented
[at this link](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html).

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.5-target-magnitude-hist.png" alt="A histogram of earthquake magnitude with a dashed grid" width="700">

<em>The distribution of magnitude across the whole catalog.</em>


Then redraw it with a logarithmic scale on the y-axis, which brings the rare large events into
view alongside the many small ones. Hint: here you can find a
[tutorial](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.yscale.html) for how to
change an axis scale.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.5-target-magnitude-hist-log.png" alt="The same histogram of earthquake magnitude with a logarithmic y-axis" width="700">

<em>The same histogram with a logarithmic count axis.</em>


Finally, make one histogram for the filtered dataset and one for the unfiltered dataset, side by
side.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.5-target-hist-filtered-unfiltered.png" alt="Two histograms of earthquake magnitude side by side, one for events above magnitude 4 and one for the whole catalog" width="100%">

<em>The filtered and unfiltered magnitude distributions, drawn on their own count axes.</em>

In [ ]:
# Make the three histograms

**Q11) Visualise the locations of earthquakes by making a scatterplot of their latitude and longitude.**

Make a figure with two panels: one for the filtered dataset and one for the unfiltered one, with
the points coloured by magnitude on a shared colour range.

Hint: consider reading the documentation for
[`plt.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) to make
the scatter plot and that of
[`plt.colorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.colorbar.html) to
colour the points by magnitude. Pass the same `vmin` and `vmax` to both scatter calls, so that a
colour stands for the same magnitude in both panels.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/1.5-target-quake-scatter.png" alt="Two scatter plots of earthquake longitude against latitude coloured by magnitude, one filtered to magnitude above 4 and one unfiltered" width="100%">

<em>Every event in the catalog placed by longitude and latitude, filtered and unfiltered.</em>

In [ ]:
# Make the two scatter plots

Do you notice a difference between the filtered and unfiltered datasets?